# FINDETECT — Block D numerical features

Run this notebook **after** `01_Block_D_Staging.ipynb` finishes
creating all three datasets under `STAGE_ROOT`. This notebook builds a receiver behavior
lookup, then writes one numerical feature Parquet file per bucket. Its output directory
is `FINDETECT_Block_D_Numerical_Final` on Drive.

**Run order:** install and mount Drive → review paths → define functions → check staging
→ build receiver lookup → run and inspect bucket 0 → run buckets 1–63 → validate all
outputs → print candidate X columns and the y label.

The upstream Parquet inputs, staging datasets, receiver lookup, and generated outputs
are external to GitHub. Use the same source files and paths to reproduce this run.
The expected total of `155,493,482` rows in the setup is a validation check for this
specific input; confirm it against your source dataset before using different data.


In [ ]:
# Install requirements and mount Google Drive
%pip install -q duckdb==1.4.1 pandas pyarrow
from google.colab import drive
drive.mount('/content/drive')


## 1. Setup

All the paths and constants here are based on the FINDETECT project layout inside Google
Drive (as produced by earlier stages of the pipeline). Edit `STAGE_ROOT` /
`RECEIVER_LOOKUP_ROOT` / `D_OUTPUT_DIR` if you're running this on different
storage.


In [ ]:
from pathlib import Path
import pandas as pd
import json

TOTAL_BUCKETS = 64
# Total expected rows across all 64 buckets combined — used in the
# final validation (the last cell of the "Final Validation" section) as an extra safety gate.
EXPECTED_TOTAL_ROWS = 155_493_482
MEMORY_LIMIT = '6GB'
THREADS = 2

# Staging that is already ready (not rebuilt here)
STAGE_ROOT = Path('/content/drive/MyDrive/processed/findetect_D_stage_v2')
# Output of the extra step: mapping receiver behavior across buckets
RECEIVER_LOOKUP_ROOT = Path('/content/drive/MyDrive/processed/findetect_D_receiver_behavior_lookup')
# Output of this notebook: one parquet file per bucket with the numerical features
D_OUTPUT_DIR = Path('/content/drive/MyDrive/processed/FINDETECT_Block_D_Numerical_Final')
# Temporary local work folder (on the Colab disk, not Drive) to speed up writing before the final copy
LOCAL_WORK = Path('/content/findetect_d_numerical_final_work')

# Columns that must pass through unchanged from staging to every output
# file, unchanged — used as a validation list (schema check) in several later cells.
REQUIRED_TRANSACTION_COLUMNS = [
    'Timestamp', 'From Bank', 'Account', 'To Bank', 'Account_duplicated_0',
    'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency',
    'Payment Format', 'Week', 'Week_Index', 'Graph_History_Week_Index',
    'Is Laundering', 'Pattern Type',
    'sender_out_degree', 'sender_in_degree', 'sender_unique_counterparties',
    'receiver_out_degree', 'receiver_in_degree', 'receiver_unique_counterparties',
    'sender_receiver_frequency', 'bidirectional_relationship',
]


## 2. Build and Validation Functions

All the functions in the next cell are defined only — no immediate
execution. The core logic is `build_bucket()`, which produces one parquet
file per bucket, with exactly the same row count as its source (does not
change the grain), then validates the output before accepting it.


In [ ]:
"""FINDETECT Block D numerical features, using existing 64-bucket staging."""
from __future__ import annotations

import gc
import json
import os
import shutil
import time
from pathlib import Path

import duckdb


LABEL_COLUMNS = ["Is Laundering", "Pattern Type"]

# Columns from earlier stages (iforest/LOF/risk_score/...) that are
# forbidden in this notebook's output, because they are later modeling
# decisions/results, not raw features.
FORBIDDEN_COLUMNS = [
    "Pattern_Type_Encoded", "Unusual_Amount_Indicator",
    "Multiple_Counterparties", "High_Transaction_Frequency",
    "Rapid_Movement_Indicator", "Structuring_Indicator",
    "iforest_score", "LOF_score", "risk_score",
    "small_tx_count_window", "small_tx_sum_window",
]

# The new numerical columns added by this notebook (see the closing
# note at the bottom for an explanation of each group).
NEW_COLUMNS = [
    "sender_total_sent_prev_week", "sender_avg_sent_prev_week",
    "sender_max_sent_prev_week", "sender_outgoing_count_prev_week",
    "receiver_total_received_prev_week", "receiver_avg_received_prev_week",
    "receiver_max_received_prev_week", "receiver_incoming_count_prev_week",
    "sender_outgoing_count_so_far_wk", "sender_unique_receivers_so_far_wk",
    "amount_vs_prev_max_diff", "amount_vs_prev_max_ratio",
    "transaction_frequency_diff", "transaction_frequency_ratio",
    "hours_since_last_incoming", "outgoing_to_incoming_ratio",
    "outgoing_count_24h", "outgoing_sum_24h",
    "outgoing_count_72h", "outgoing_sum_72h",
]

# Behavior column names may differ between staging versions, so each
# required feature has a list of candidate names, and
# resolve_behavior_schema tries to find the first one that actually exists.
BEHAVIOR_CANDIDATES = {
    "sender_total_sent_prev_week": ["total_amount_sent_prev_week", "sender_total_sent_prev_week", "total_sent_prev_week"],
    "sender_avg_sent_prev_week": ["avg_amount_sent_prev_week", "sender_avg_sent_prev_week", "avg_sent_prev_week"],
    "sender_max_sent_prev_week": ["max_amount_sent_prev_week", "sender_max_sent_prev_week", "max_sent_prev_week"],
    "sender_outgoing_count_prev_week": ["outgoing_count_prev_week", "sender_outgoing_count_prev_week"],
    "receiver_total_received_prev_week": ["total_amount_received_prev_week", "receiver_total_received_prev_week", "total_received_prev_week"],
    "receiver_avg_received_prev_week": ["avg_amount_received_prev_week", "receiver_avg_received_prev_week", "avg_received_prev_week"],
    "receiver_max_received_prev_week": ["max_amount_received_prev_week", "receiver_max_received_prev_week", "max_received_prev_week"],
    "receiver_incoming_count_prev_week": ["incoming_count_prev_week", "receiver_incoming_count_prev_week"],
}

# Ratio columns that could produce inf when dividing by zero despite
# NULLIF — checked after every build (per bucket) and in the final
# validation, via count_non_finite_ratios() below instead of repeating
# the same SQL statement everywhere.
RATIO_COLUMNS_TO_VALIDATE = [
    "amount_vs_prev_max_ratio",
    "transaction_frequency_ratio",
    "outgoing_to_incoming_ratio",
]


def qi(name):
    """Quote an identifier (column/table name) for safe use in SQL."""
    return '"' + str(name).replace('"', '""') + '"'


def qs(value):
    """Quote a value as a SQL string literal."""
    return "'" + str(value).replace("'", "''") + "'"


def new_connection(memory_limit, threads, temp_directory):
    """Open a fresh DuckDB connection configured with the run's memory
    limit, thread count, and a scratch directory for spill-to-disk."""
    Path(temp_directory).mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.execute(f"SET memory_limit={qs(memory_limit)}")
    con.execute(f"SET threads={int(threads)}")
    con.execute("SET preserve_insertion_order=false")
    con.execute("SET enable_progress_bar=true")
    con.execute(f"SET temp_directory={qs(temp_directory)}")
    return con


def bucket_files(stage_dir, bucket_id):
    """List the parquet files for one bucket under a Hive-partitioned
    (d_bucket_id=N) staging folder. Raises if none are found."""
    stage_dir = Path(stage_dir)
    matches = []
    for folder in stage_dir.glob("d_bucket_id=*") if stage_dir.exists() else []:
        try:
            if int(folder.name.split("=", 1)[1]) == int(bucket_id):
                matches.extend(folder.glob("*.parquet"))
        except ValueError:
            continue
    files = sorted({p.resolve() for p in matches})
    if not files:
        raise FileNotFoundError(f"No Parquet files for bucket {bucket_id} under {stage_dir}")
    return files


def parquet_list(files, hive=False):
    """Build a DuckDB read_parquet() expression over an explicit list
    of files."""
    values = ",".join(qs(str(p)) for p in files)
    return f"read_parquet([{values}], union_by_name=true, hive_partitioning={'true' if hive else 'false'})"


def parquet_glob(path, hive=True):
    """Build a DuckDB read_parquet() expression over a glob/path."""
    return f"read_parquet({qs(str(path))}, union_by_name=true, hive_partitioning={'true' if hive else 'false'})"


def relation_columns(con, relation):
    """Return the column names of a DuckDB relation/SQL expression
    without materializing any rows."""
    return [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM {relation} LIMIT 0").fetchall()]


def require_columns(found, required, context):
    """Raise if any of `required` is missing from `found`, naming the
    context (e.g. which check) in the error for easier debugging."""
    missing = sorted(set(required) - set(found))
    if missing:
        raise RuntimeError(f"{context}: missing columns {missing}")


def resolve(found, candidates, description):
    """Return the first name in `candidates` that is present in
    `found`; raise a descriptive error if none match."""
    for candidate in candidates:
        if candidate in found:
            return candidate
    raise RuntimeError(f"Cannot resolve {description}. Tried {candidates}. Found {found}")


def resolve_behavior_schema(con, behavior_relation):
    """Resolve every BEHAVIOR_CANDIDATES entry against the actual
    columns of the behavior relation, returning {output_name: source_name}."""
    found = relation_columns(con, behavior_relation)
    require_columns(found, ["Final_Account", "Week"], "behavior")
    return {out: resolve(found, candidates, out) for out, candidates in BEHAVIOR_CANDIDATES.items()}


def count_non_finite_ratios(con, relation):
    """Count rows where any RATIO_COLUMNS_TO_VALIDATE value is non-null
    but not finite (e.g. inf that NULLIF's zero-guard didn't catch).
    Shared by the per-bucket build check, the pilot check, and the
    final delivery check, so the three stay in sync."""
    condition = " OR ".join(
        f"({qi(c)} IS NOT NULL AND NOT isfinite({qi(c)}))"
        for c in RATIO_COLUMNS_TO_VALIDATE
    )
    return con.execute(f"SELECT count(*) FROM {relation} WHERE {condition}").fetchone()[0]


def inspect_staging(stage_root, total_buckets=64):
    """Check that the three expected staging datasets (sender
    transactions, incoming events, behavior) each have exactly
    `total_buckets` partitions with at least one file per partition.
    Raises if the staging is incomplete; never modifies it."""
    root = Path(stage_root)
    report = []
    for name in ["sender_transactions", "incoming_events", "behavior"]:
        folder = root / name
        partitions = list(folder.glob("d_bucket_id=*")) if folder.exists() else []
        files = list(folder.glob("d_bucket_id=*/*.parquet")) if folder.exists() else []
        report.append({
            "dataset": name,
            "folder_exists": folder.exists(),
            "partitions": len(partitions),
            "parquet_files": len(files),
            "ready": folder.exists() and len(partitions) == total_buckets and len(files) >= total_buckets,
        })
    if not all(x["ready"] for x in report):
        raise RuntimeError(f"STAGING IS NOT COMPLETE: {report}")
    return report


def _safe_aux_path(path):
    """Guard rail for build_receiver_behavior_lookup(), which deletes
    and rebuilds this path from scratch: refuse to operate on a path
    that doesn't look like one of this project's own auxiliary
    directories, to avoid an accidental rmtree() somewhere unrelated."""
    path = Path(path)
    if "findetect" not in str(path).lower() or len(path.parts) < 4:
        raise RuntimeError(f"Unsafe auxiliary path: {path}")
    return path


def receiver_lookup_valid(aux_root, total_buckets=64):
    """Check whether a previously built receiver-behavior lookup at
    `aux_root` is complete for `total_buckets`, using its _SUCCESS.json
    marker plus an independent recount of partitions/files."""
    root = Path(aux_root)
    marker = root / "_SUCCESS.json"
    if not marker.exists():
        return False
    try:
        meta = json.loads(marker.read_text(encoding="utf-8"))
        partitions = list(root.glob("d_bucket_id=*"))
        files = list(root.glob("d_bucket_id=*/*.parquet"))
        return meta.get("total_buckets") == total_buckets and len(partitions) == total_buckets and len(files) >= total_buckets
    except Exception:
        return False


def build_receiver_behavior_lookup(stage_root, aux_root, total_buckets, memory_limit, threads, local_work):
    """Map receiver previous-week features to the sender's bucket once.

    A transaction's sender and receiver can fall in different behavior
    buckets, so a plain per-bucket join would miss the receiver's own
    behavior features. This builds one lookup table
    (bucket, receiver_account, receiver_week) -> receiver behavior
    features, keyed off every distinct receiver seen in the sender
    staging, and writes it as a checkpoint under `aux_root`. If a valid
    checkpoint already exists it is reused (skipped) instead of rebuilt.
    Never modifies the original staging.
    """
    if receiver_lookup_valid(aux_root, total_buckets):
        print("Receiver behavior lookup: valid — skipped")
        return Path(aux_root)

    aux_root = _safe_aux_path(aux_root)
    if aux_root.exists():
        shutil.rmtree(aux_root)
    aux_root.mkdir(parents=True, exist_ok=True)

    stage_root = Path(stage_root)
    sender = parquet_glob(stage_root / "sender_transactions" / "d_bucket_id=*" / "*.parquet", hive=True)
    behavior = parquet_glob(stage_root / "behavior" / "d_bucket_id=*" / "*.parquet", hive=True)
    con = new_connection(memory_limit, threads, Path(local_work) / "receiver_lookup_tmp")
    started = time.time()
    try:
        sender_cols = relation_columns(con, sender)
        require_columns(sender_cols, ["d_bucket_id", "Account_duplicated_0", "Week"], "sender staging")
        mapping = resolve_behavior_schema(con, behavior)

        duplicate = con.execute(f"""
            SELECT 1 FROM {behavior}
            GROUP BY {qi('Final_Account')}, {qi('Week')}
            HAVING count(*) > 1 LIMIT 1
        """).fetchone()
        if duplicate:
            raise RuntimeError("behavior Final_Account + Week is not unique")

        receiver_projection = ",\n".join(
            f"CAST(b.{qi(mapping[name])} AS DOUBLE) AS {qi(name)}"
            for name in [
                "receiver_total_received_prev_week", "receiver_avg_received_prev_week",
                "receiver_max_received_prev_week", "receiver_incoming_count_prev_week",
            ]
        )
        sql = f"""
        WITH receiver_keys AS (
            SELECT DISTINCT
                CAST(d_bucket_id AS INTEGER) AS d_bucket_id,
                {qi('Account_duplicated_0')} AS receiver_account,
                {qi('Week')} AS receiver_week
            FROM {sender}
            WHERE {qi('Account_duplicated_0')} IS NOT NULL
        )
        SELECT
            k.d_bucket_id,
            k.receiver_account,
            k.receiver_week,
            {receiver_projection}
        FROM receiver_keys k
        LEFT JOIN {behavior} b
          ON k.receiver_account = b.{qi('Final_Account')}
         AND k.receiver_week = b.{qi('Week')}
        """
        con.execute(
            f"COPY ({sql}) TO {qs(aux_root)} "
            "(FORMAT PARQUET, COMPRESSION ZSTD, PARTITION_BY(d_bucket_id), ROW_GROUP_SIZE 100000)"
        )
    finally:
        con.close()

    partitions = list(aux_root.glob("d_bucket_id=*"))
    files = list(aux_root.glob("d_bucket_id=*/*.parquet"))
    if len(partitions) != total_buckets or len(files) < total_buckets:
        raise RuntimeError(f"Receiver lookup incomplete: partitions={len(partitions)}, files={len(files)}")
    marker = {"total_buckets": total_buckets, "minutes": round((time.time() - started) / 60, 2)}
    (aux_root / "_SUCCESS.json").write_text(json.dumps(marker, indent=2), encoding="utf-8")
    print(f"Receiver behavior lookup: complete in {marker['minutes']} min")
    return aux_root


def output_path(output_dir, bucket_id):
    """Standard output path for one bucket's numerical-features parquet file."""
    return Path(output_dir) / f"block_d_numerical_bucket_{int(bucket_id):03d}.parquet"


def checkpoint_valid(output_dir, bucket_id, required_transaction_columns, memory_limit, threads, local_work):
    """Check whether a bucket's output file already exists, has the
    required + new columns and none of the forbidden ones, has at
    least one row, and is entirely tagged with the right d_bucket_id.
    Used to make build_bucket() and the final delivery check
    resumable/idempotent."""
    path = output_path(output_dir, bucket_id)
    if not path.exists() or path.stat().st_size == 0:
        return False
    con = None
    try:
        con = new_connection(memory_limit, threads, Path(local_work) / "checkpoint_tmp")
        rel = parquet_glob(path, hive=False)
        found = relation_columns(con, rel)
        require_columns(found, required_transaction_columns + NEW_COLUMNS + ["d_bucket_id"], "checkpoint")
        if set(found) & set(FORBIDDEN_COLUMNS):
            return False
        row = con.execute(f"SELECT count(*), min(d_bucket_id), max(d_bucket_id) FROM {rel}").fetchone()
        return row[0] > 0 and row[1] == int(bucket_id) and row[2] == int(bucket_id)
    except Exception:
        return False
    finally:
        if con is not None:
            con.close()


def build_bucket(bucket_id, stage_root, receiver_lookup_root, output_dir,
                 required_transaction_columns, memory_limit="6GB", threads=2,
                 local_work="/content/findetect_numerical_work", force=False):
    """Build the numerical-features parquet file for one bucket.

    Pipeline: passthrough the original transaction columns unchanged,
    add rolling/window features computed over the sender's own history
    (same-week running counts, 24h/72h outgoing activity), ASOF-join the
    most recent incoming transaction per account, left-join the sender's
    own previous-week behavior and the receiver's previous-week behavior
    (via the receiver lookup built earlier), then derive the ratio/diff
    features from those joined columns. Writes to a local scratch file
    first, validates row count/grain/schema/ratio-finiteness there, then
    copies to Drive and validates the copy before the atomic rename to
    the final path. If a valid checkpoint already exists for this bucket
    and `force` is False, the bucket is skipped entirely.
    """
    bucket_id = int(bucket_id)
    if not force and checkpoint_valid(output_dir, bucket_id, required_transaction_columns, memory_limit, threads, local_work):
        print(f"Bucket {bucket_id:03d}: valid — skipped")
        return output_path(output_dir, bucket_id)

    stage_root = Path(stage_root)
    sender_files = bucket_files(stage_root / "sender_transactions", bucket_id)
    incoming_files = bucket_files(stage_root / "incoming_events", bucket_id)
    behavior_files = bucket_files(stage_root / "behavior", bucket_id)
    receiver_files = bucket_files(receiver_lookup_root, bucket_id)
    sender = parquet_list(sender_files)
    incoming = parquet_list(incoming_files)
    behavior = parquet_list(behavior_files)
    receiver_lookup = parquet_list(receiver_files)

    final = output_path(output_dir, bucket_id)
    final.parent.mkdir(parents=True, exist_ok=True)
    work = Path(local_work) / f"bucket_{bucket_id:03d}"
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)
    local = work / final.name
    partial = final.with_suffix(final.suffix + ".partial")
    if partial.exists():
        partial.unlink()

    con = new_connection(memory_limit, threads, work / "duckdb_tmp")
    started = time.time()
    try:
        sender_cols = relation_columns(con, sender)
        require_columns(sender_cols, required_transaction_columns, "sender bucket")
        incoming_cols = relation_columns(con, incoming)
        incoming_account = resolve(incoming_cols, ["incoming_account", "Final_Account", "Account"], "incoming account")
        incoming_ts = resolve(incoming_cols, ["incoming_ts", "incoming_timestamp", "Timestamp"], "incoming timestamp")
        incoming_amount = resolve(incoming_cols, ["incoming_amount", "Amount Received", "amount"], "incoming amount")
        mapping = resolve_behavior_schema(con, behavior)

        duplicate = con.execute(f"""
            SELECT 1 FROM {behavior}
            GROUP BY {qi('Final_Account')}, {qi('Week')}
            HAVING count(*) > 1 LIMIT 1
        """).fetchone()
        if duplicate:
            raise RuntimeError(f"Bucket {bucket_id}: sender behavior keys are duplicated")

        bad_graph = con.execute(f"""
            SELECT count(*) FROM {sender}
            WHERE {qi('Graph_History_Week_Index')} IS NOT NULL
              AND {qi('Graph_History_Week_Index')} <> {qi('Week_Index')} - 1
        """).fetchone()[0]
        if bad_graph:
            raise RuntimeError(f"Bucket {bucket_id}: graph history failed for {bad_graph:,} rows")

        source_rows = con.execute(f"SELECT count(*) FROM {sender}").fetchone()[0]
        recomputed = set(NEW_COLUMNS + FORBIDDEN_COLUMNS + ["d_bucket_id", "_source_row_id"])
        passthrough = [c for c in sender_cols if c not in recomputed]
        passthrough_sql = ",\n".join(f"s.{qi(c)}" for c in passthrough)
        sender_projection = ",\n".join(
            f"CAST(b.{qi(mapping[name])} AS DOUBLE) AS {qi(name)}"
            for name in [
                "sender_total_sent_prev_week", "sender_avg_sent_prev_week",
                "sender_max_sent_prev_week", "sender_outgoing_count_prev_week",
            ]
        )

        sql = f"""
        WITH source AS MATERIALIZED (
            SELECT {passthrough_sql}, {bucket_id}::INTEGER AS d_bucket_id
            FROM {sender} s
        ),
        running AS (
            SELECT s.*,
                count(*) OVER (
                    PARTITION BY {qi('Account')}, {qi('Week')}
                    ORDER BY {qi('Timestamp')}
                    RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                )::BIGINT AS sender_outgoing_count_so_far_wk,
                count(DISTINCT {qi('Account_duplicated_0')}) OVER (
                    PARTITION BY {qi('Account')}, {qi('Week')}
                    ORDER BY {qi('Timestamp')}
                    RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                )::BIGINT AS sender_unique_receivers_so_far_wk,
                count(*) OVER (
                    PARTITION BY {qi('Account')} ORDER BY {qi('Timestamp')}
                    RANGE BETWEEN INTERVAL 24 HOUR PRECEDING AND CURRENT ROW
                )::BIGINT AS outgoing_count_24h,
                sum(CAST({qi('Amount Paid')} AS DOUBLE)) OVER (
                    PARTITION BY {qi('Account')} ORDER BY {qi('Timestamp')}
                    RANGE BETWEEN INTERVAL 24 HOUR PRECEDING AND CURRENT ROW
                )::DOUBLE AS outgoing_sum_24h,
                count(*) OVER (
                    PARTITION BY {qi('Account')} ORDER BY {qi('Timestamp')}
                    RANGE BETWEEN INTERVAL 72 HOUR PRECEDING AND CURRENT ROW
                )::BIGINT AS outgoing_count_72h,
                sum(CAST({qi('Amount Paid')} AS DOUBLE)) OVER (
                    PARTITION BY {qi('Account')} ORDER BY {qi('Timestamp')}
                    RANGE BETWEEN INTERVAL 72 HOUR PRECEDING AND CURRENT ROW
                )::DOUBLE AS outgoing_sum_72h
            FROM source s
        ),
        incoming_agg AS (
            SELECT
                {qi(incoming_account)} AS incoming_account,
                CAST({qi(incoming_ts)} AS TIMESTAMP) AS incoming_timestamp,
                sum(CAST({qi(incoming_amount)} AS DOUBLE))::DOUBLE AS last_incoming_amount
            FROM {incoming}
            GROUP BY 1, 2
        ),
        with_incoming AS (
            SELECT r.*, i.incoming_timestamp, i.last_incoming_amount
            FROM running r
            ASOF LEFT JOIN incoming_agg i
              ON r.{qi('Account')} = i.incoming_account
             AND r.{qi('Timestamp')} >= i.incoming_timestamp
        ),
        with_sender AS (
            SELECT x.*, {sender_projection}
            FROM with_incoming x
            LEFT JOIN {behavior} b
              ON x.{qi('Account')} = b.{qi('Final_Account')}
             AND x.{qi('Week')} = b.{qi('Week')}
        ),
        joined AS (
            SELECT s.*,
                r.receiver_total_received_prev_week,
                r.receiver_avg_received_prev_week,
                r.receiver_max_received_prev_week,
                r.receiver_incoming_count_prev_week
            FROM with_sender s
            LEFT JOIN {receiver_lookup} r
              ON s.{qi('Account_duplicated_0')} = r.receiver_account
             AND s.{qi('Week')} = r.receiver_week
        )
        SELECT
            * EXCLUDE (incoming_timestamp, last_incoming_amount),
            CAST({qi('Amount Paid')} AS DOUBLE) - sender_max_sent_prev_week AS amount_vs_prev_max_diff,
            CAST({qi('Amount Paid')} AS DOUBLE) / NULLIF(sender_max_sent_prev_week, 0) AS amount_vs_prev_max_ratio,
            sender_outgoing_count_so_far_wk - sender_outgoing_count_prev_week AS transaction_frequency_diff,
            sender_outgoing_count_so_far_wk::DOUBLE / NULLIF(sender_outgoing_count_prev_week, 0) AS transaction_frequency_ratio,
            date_diff('microsecond', incoming_timestamp, CAST({qi('Timestamp')} AS TIMESTAMP)) / 3600000000.0 AS hours_since_last_incoming,
            CAST({qi('Amount Paid')} AS DOUBLE) / NULLIF(last_incoming_amount, 0) AS outgoing_to_incoming_ratio
        FROM joined
        """
        print(f"Bucket {bucket_id:03d}: running...")
        con.execute(
            f"COPY ({sql}) TO {qs(local)} "
            "(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 250000)"
        )
        local_rel = parquet_glob(local, hive=False)
        result = con.execute(f"SELECT count(*), min(d_bucket_id), max(d_bucket_id) FROM {local_rel}").fetchone()
        if result != (source_rows, bucket_id, bucket_id):
            raise RuntimeError(f"Bucket {bucket_id}: row/grain validation failed {result}, expected {source_rows}")
        found = relation_columns(con, local_rel)
        require_columns(found, required_transaction_columns + NEW_COLUMNS + ["d_bucket_id"], "local output")
        forbidden = sorted(set(found) & set(FORBIDDEN_COLUMNS))
        if forbidden:
            raise RuntimeError(f"Forbidden columns in output: {forbidden}")
        invalid = count_non_finite_ratios(con, local_rel)
        if invalid:
            raise RuntimeError(f"Bucket {bucket_id}: {invalid} non-finite ratio rows")
    finally:
        con.close()

    shutil.copy2(local, partial)
    verify = duckdb.connect()
    try:
        count = verify.execute(f"SELECT count(*) FROM {parquet_glob(partial, hive=False)}").fetchone()[0]
        if count != source_rows:
            raise RuntimeError(f"Drive partial copy has {count} rows, expected {source_rows}")
    finally:
        verify.close()
    os.replace(partial, final)
    shutil.rmtree(work)
    gc.collect()
    print(f"Bucket {bucket_id:03d}: saved {source_rows:,} rows in {(time.time()-started)/60:.2f} min")
    return final


def run_buckets(buckets, **kwargs):
    """Build a sequence of buckets, stopping at the first failure. This
    is what makes both the pilot ([0]) and the full run
    (range(1, TOTAL_BUCKETS)) go through the same code path."""
    results = []
    for bucket_id in buckets:
        try:
            path = build_bucket(bucket_id, **kwargs)
            results.append({"bucket": int(bucket_id), "status": "ok", "path": str(path)})
        except Exception as exc:
            print(f"Bucket {int(bucket_id):03d}: FAILED — {exc}")
            results.append({"bucket": int(bucket_id), "status": "failed", "error": repr(exc)})
            break
        finally:
            gc.collect()
    return results


## 3. Check the Existing Staging — Does Not Rebuild It

In [ ]:
staging_report = inspect_staging(STAGE_ROOT, TOTAL_BUCKETS)
display(pd.DataFrame(staging_report))
print('STAGING IS COMPLETE — it will not be rebuilt')


## 4. Build the Receiver Behavior Lookup Once

This is a small but necessary addition, because the Receiver may fall into a
behavior bucket different from the Sender's. If it was already completed
previously, it will show `valid — skipped`. Does not modify the original
Staging.


In [ ]:
build_receiver_behavior_lookup(
    STAGE_ROOT, RECEIVER_LOOKUP_ROOT, TOTAL_BUCKETS,
    MEMORY_LIMIT, THREADS, LOCAL_WORK
)


## 5. Pilot — Bucket 0 Only

In [ ]:
common_args = dict(
    stage_root=STAGE_ROOT, receiver_lookup_root=RECEIVER_LOOKUP_ROOT,
    output_dir=D_OUTPUT_DIR, required_transaction_columns=REQUIRED_TRANSACTION_COLUMNS,
    memory_limit=MEMORY_LIMIT, threads=THREADS, local_work=LOCAL_WORK
)
pilot_result = run_buckets([0], **common_args)
display(pd.DataFrame(pilot_result))


## 6. Check Bucket 0

Before continuing to the remaining 63 buckets, manually verify that bucket 0
did not change the row count (grain), that the forbidden columns are not
present, and that the distribution of the new values (null ratio and
statistics) is reasonable.


In [ ]:
pilot_path = output_path(D_OUTPUT_DIR, 0)
con = new_connection(MEMORY_LIMIT, THREADS, LOCAL_WORK / 'pilot_check_tmp')
try:
    rel = parquet_glob(pilot_path, hive=False)
    source_rel = parquet_list(bucket_files(STAGE_ROOT / 'sender_transactions', 0))
    output_rows = con.execute(f'SELECT count(*) FROM {rel}').fetchone()[0]
    source_rows = con.execute(f'SELECT count(*) FROM {source_rel}').fetchone()[0]
    if output_rows != source_rows:
        raise RuntimeError(f'Join changed grain: source={source_rows}, output={output_rows}')
    schema = con.execute(f'DESCRIBE SELECT * FROM {rel}').df()
    names = schema.column_name.tolist()
    forbidden = sorted(set(names) & set(FORBIDDEN_COLUMNS))
    if forbidden: raise RuntimeError(f'Forbidden columns: {forbidden}')
    require_columns(names, REQUIRED_TRANSACTION_COLUMNS + NEW_COLUMNS + ['d_bucket_id'], 'pilot')
    display(con.execute(f'SELECT * FROM {rel} LIMIT 5').df())
    display(schema)
    null_sql = ','.join([f'avg(({qi(c)} IS NULL)::INT) AS {qi(c)}' for c in NEW_COLUMNS])
    nulls = con.execute(f'SELECT {null_sql} FROM {rel}').df().T.reset_index()
    nulls.columns = ['feature','null_ratio']; display(nulls)
    stats=[]
    for c in NEW_COLUMNS:
        q=qi(c)
        stats.append(f"SELECT {qs(c)} feature,min({q}) minimum,median({q}) median,quantile_cont({q},.90) p90,quantile_cont({q},.95) p95,quantile_cont({q},.99) p99,max({q}) maximum FROM {rel}")
    display(con.execute(' UNION ALL '.join(stats)).df())
    invalid = count_non_finite_ratios(con, rel)
    if invalid: raise RuntimeError(f'Non-finite ratios: {invalid}')
    print(f'Bucket 0 rows: {output_rows:,}')
    print('PILOT VALIDATION PASSED')
finally:
    con.close()


## 7. Run the remaining 63 buckets

After the pilot succeeds, this cell processes buckets 1–63. Valid existing bucket
outputs are skipped, so rerun it after a Colab disconnection if needed.


In [ ]:
full_results = run_buckets(range(1, TOTAL_BUCKETS), **common_args)
display(pd.DataFrame(full_results).tail())


## 8. Final Validation

Validates all 64 files together: no missing files, every checkpoint
individually valid, no forbidden columns, total row count matches
`EXPECTED_TOTAL_ROWS`, every bucket present exactly once, and no rows with
non-finite ratios.


In [ ]:
expected = [output_path(D_OUTPUT_DIR, i) for i in range(TOTAL_BUCKETS)]
missing = [p.name for p in expected if not p.exists()]
if missing: raise RuntimeError(f'Missing output files: {missing}')
invalid = [i for i in range(TOTAL_BUCKETS) if not checkpoint_valid(D_OUTPUT_DIR,i,REQUIRED_TRANSACTION_COLUMNS,MEMORY_LIMIT,THREADS,LOCAL_WORK)]
if invalid: raise RuntimeError(f'Invalid checkpoints: {invalid}')
con = new_connection(MEMORY_LIMIT, THREADS, LOCAL_WORK / 'final_check_tmp')
try:
    rel = parquet_glob(D_OUTPUT_DIR / 'block_d_numerical_bucket_*.parquet', hive=False)
    names = relation_columns(con, rel)
    require_columns(names, REQUIRED_TRANSACTION_COLUMNS + NEW_COLUMNS + ['d_bucket_id'], 'full output')
    bad = sorted(set(names) & set(FORBIDDEN_COLUMNS))
    if bad: raise RuntimeError(f'Forbidden columns: {bad}')
    total = con.execute(f'SELECT count(*) FROM {rel}').fetchone()[0]
    if total != EXPECTED_TOTAL_ROWS: raise RuntimeError(f'Row mismatch: {total:,} != {EXPECTED_TOTAL_ROWS:,}')
    buckets = con.execute(f'SELECT d_bucket_id, count(*) AS row_count FROM {rel} GROUP BY 1 ORDER BY 1').df()
    if buckets.d_bucket_id.tolist() != list(range(TOTAL_BUCKETS)): raise RuntimeError('Bucket ids are incomplete')
    invalid_ratios = count_non_finite_ratios(con, rel)
    if invalid_ratios: raise RuntimeError(f'Non-finite ratios: {invalid_ratios}')
    display(buckets)
    print(f'Files: {len(expected)}, rows: {total:,}')
    print('FULL NUMERICAL BLOCK D DELIVERY VALIDATION PASSED')
finally:
    con.close()


## 9. X and y Lists — Names Only, No Training

The next cell only builds the list of numerical column names that are
candidates for X (excluding the labels, the identity/time columns, and the
forbidden columns), and sets the y column. There is no training or
Train/Test split here.


In [ ]:
EXCLUDED_FROM_X = set(LABEL_COLUMNS + ['Timestamp','From Bank','Account','To Bank','Account_duplicated_0','Payment Format','Payment Currency','Receiving Currency','d_bucket_id','Week','Week_Index','Graph_History_Week_Index'])
con = new_connection(MEMORY_LIMIT, THREADS, LOCAL_WORK / 'schema_tmp')
try:
    rel = parquet_glob(D_OUTPUT_DIR / 'block_d_numerical_bucket_*.parquet', hive=False)
    schema = con.execute(f'DESCRIBE SELECT * FROM {rel}').df()
finally:
    con.close()
numeric_prefixes=('TINYINT','SMALLINT','INTEGER','BIGINT','HUGEINT','FLOAT','DOUBLE','DECIMAL','UTINYINT','USMALLINT','UINTEGER','UBIGINT')
X_COLUMNS=[r.column_name for r in schema.itertuples() if r.column_name not in EXCLUDED_FROM_X and r.column_name not in FORBIDDEN_COLUMNS and str(r.column_type).upper().startswith(numeric_prefixes)]
y_COLUMN='Is Laundering'
print('X columns:', X_COLUMNS)
print('y:', y_COLUMN)
print('Auxiliary label excluded:', 'Pattern Type')


## The New Numerical Columns

- 8 behavioral features from the previous week for the sender and receiver.
- The sender's cumulative activity up to the transaction (within the same week).
- Difference and ratio of the amount versus the previous highest amount.
- Difference and ratio of frequency versus the previous week.
- Hours since the last incoming transaction and the outgoing-to-incoming ratio.
- Count and sum of outgoing transactions over 24 and 72 hours.

There are no Thresholds or Risk Flags in this notebook.
